# Chess Opening Meta-Learning from Move 5 Onward

This notebook is a research prototype for next-move prediction in chess openings. It compares two training regimes on the same board encoder and move-classification head:

- Standard supervised learning: train one model directly on labeled positions from the training openings.
- MAML, short for Model-Agnostic Meta-Learning: train one initialization that should adapt quickly to a new opening after seeing a small support set.

The central experimental rule is that every prediction example starts at chess fullmove 5 or later. In zero-based ply notation this means ply 8 or later. Ply 8 is White to move on move 5, and ply 9 is Black to move on move 5. Earlier positions, such as the board before Black's first move at ply 1, are deliberately excluded because they make the task closer to memorizing opening prefixes than adapting from meaningful mid-opening structure.

The notebook is organized as follows: data loading, move vocabulary and board encoding, move-5 example construction, task splitting, model definition, supervised training, MAML training, few-shot evaluation, visualizations, and interpretation.

In [ ]:
# Colab dependency setup.
# If you are running locally and already have these packages, this cell can be skipped.
!pip -q install datasets python-chess pandas matplotlib tqdm

## Experiment Design and Move Indexing

A single labeled example is the board state before a target move, plus the target move in UCI notation. The notebook uses python-chess board states so legal moves, side to move, castling rights, en passant rights, halfmove clock, and fullmove number are all internally consistent.

Move indexing is the most important bookkeeping detail in this project:

- ply_idx is zero-based and counts half-moves already played before the target move.
- ply 0 is White's first move, written as 1.
- ply 1 is Black's first move, written as 1...
- ply 8 is White's fifth move, written as 5.
- ply 9 is Black's fifth move, written as 5...

Therefore, starting from fullmove 5 means keeping only examples with ply_idx >= 8. Both White-to-move and Black-to-move examples on move 5 are allowed, and all later plies are allowed up to max_plies. The code stores both ply_idx and target_fullmove_number in every example and validates that the target move is legal in the stored FEN.

In [ ]:
import copy
import io
import math
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import chess
import chess.pgn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from IPython.display import display
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    from torch.func import functional_call
except ImportError:
    from torch.nn.utils.stateless import functional_call

try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('default')
plt.rcParams.update(
    {
        'figure.dpi': 120,
        'savefig.dpi': 160,
        'axes.titlesize': 12,
        'axes.labelsize': 10,
        'legend.fontsize': 9,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'lines.linewidth': 2.0,
        'axes.spines.top': False,
        'axes.spines.right': False,
    }
)


def first_ply_for_fullmove(fullmove_number: int) -> int:
    if fullmove_number < 1:
        raise ValueError('fullmove_number must be at least 1')
    return 2 * (fullmove_number - 1)


def fullmove_for_ply(ply_idx: int) -> int:
    return ply_idx // 2 + 1


def side_to_move_for_ply(ply_idx: int) -> str:
    return 'white' if ply_idx % 2 == 0 else 'black'


def ply_label(ply_idx: int) -> str:
    suffix = '.' if ply_idx % 2 == 0 else '...'
    return f'{fullmove_for_ply(ply_idx)}{suffix}'


@dataclass
class Config:
    data_source: str = 'huggingface'  # one of: huggingface, kaggle_csv, pgn_file
    kaggle_csv_path: str = '/content/chess-openings.csv'
    pgn_path: str = '/content/lichess_sample.pgn'
    max_games_from_pgn: int = 5000

    task_group_field: str = 'name'  # use 'name' for fine-grained openings, or 'eco' for coarser tasks
    holdout_task: Optional[str] = None  # exact opening name or ECO code; None chooses a large eligible task
    validation_task_count: int = 8

    max_plies: int = 16
    min_prediction_fullmove: int = 5
    min_examples_per_task: int = 8
    support_size: int = 4
    query_size: int = 4
    num_eval_episodes: int = 30

    batch_size: int = 128
    baseline_epochs: int = 6
    baseline_lr: float = 3e-4
    baseline_log_every_batches: int = 20

    meta_steps: int = 250
    meta_batch_size: int = 8
    outer_lr: float = 5e-4
    inner_lr: float = 0.1
    inner_steps: int = 2
    maml_log_every_steps: int = 25
    grad_clip_norm: float = 1.0

    adaptation_steps: int = 12
    adaptation_lr: float = 5e-4

    d_model: int = 128
    num_heads: int = 4
    num_layers: int = 3
    ff_dim: int = 256
    dropout: float = 0.1

    @property
    def min_prediction_ply(self) -> int:
        return first_ply_for_fullmove(self.min_prediction_fullmove)


cfg = Config()
if cfg.max_plies <= cfg.min_prediction_ply:
    raise ValueError('max_plies must include at least one target move at or after the configured fullmove.')
if cfg.min_examples_per_task < cfg.support_size + cfg.query_size:
    raise ValueError('min_examples_per_task must be at least support_size + query_size.')

print('Minimum prediction fullmove:', cfg.min_prediction_fullmove)
print('Minimum zero-based target ply:', cfg.min_prediction_ply)
print('First included target move label:', ply_label(cfg.min_prediction_ply))
cfg

## Dataset Pipeline

The default source is the public Lichess chess-openings dataset from Hugging Face. Each row is an opening line with metadata such as ECO code, opening name, PGN/SAN text, and UCI moves when available. The notebook can also read a Kaggle-style CSV or a raw PGN file.

The pipeline converts every line to a sequence of UCI moves. If a UCI column exists, it is used directly. Otherwise, PGN/SAN text is parsed with python-chess, with a fallback cleaner for compact tokens such as 1.e4 or 1...c5. Rows with illegal or unparsable move sequences are skipped rather than silently repaired.

The label for a position is the next move from the line. This means the board must be encoded before pushing the target move. The example-construction cell below enforces that order explicitly and later validates the stored FEN against the stored target_uci.

In [ ]:
RESULT_TOKENS = {'1-0', '0-1', '1/2-1/2', '*'}
COMMENT_RE = re.compile(r'\{[^}]*\}')
VARIATION_RE = re.compile(r'\([^()]*\)')
NAG_RE = re.compile(r'\$\d+')


def load_hf_openings_dataframe() -> pd.DataFrame:
    dataset = load_dataset('Lichess/chess-openings', split='train')
    return dataset.to_pandas()


def load_kaggle_csv_dataframe(csv_path: str) -> pd.DataFrame:
    return pd.read_csv(csv_path)


def load_pgn_games_dataframe(pgn_path: str, max_games: int, max_plies: int) -> pd.DataFrame:
    records = []
    with open(pgn_path, 'r', encoding='utf-8', errors='ignore') as handle:
        while len(records) < max_games:
            game = chess.pgn.read_game(handle)
            if game is None:
                break

            opening_name = game.headers.get('Opening', 'Unknown Opening')
            eco = game.headers.get('ECO', 'UNK')
            board = game.board()
            uci_moves = []
            san_moves = []

            for ply_idx, move in enumerate(game.mainline_moves()):
                if ply_idx >= max_plies:
                    break
                san_moves.append(board.san(move))
                uci_moves.append(move.uci())
                board.push(move)

            if len(uci_moves) <= cfg.min_prediction_ply:
                continue

            records.append(
                {
                    'eco': eco,
                    'name': opening_name,
                    'pgn': ' '.join(san_moves),
                    'uci': ' '.join(uci_moves),
                    'epd': board.epd(),
                }
            )

    return pd.DataFrame(records)


def load_source_dataframe(cfg: Config) -> pd.DataFrame:
    if cfg.data_source == 'huggingface':
        return load_hf_openings_dataframe()
    if cfg.data_source == 'kaggle_csv':
        return load_kaggle_csv_dataframe(cfg.kaggle_csv_path)
    if cfg.data_source == 'pgn_file':
        return load_pgn_games_dataframe(cfg.pgn_path, cfg.max_games_from_pgn, cfg.max_plies)
    raise ValueError(f'Unsupported data_source: {cfg.data_source}')


def pgn_text_to_uci_moves_with_reader(pgn_text: str) -> List[str]:
    game = chess.pgn.read_game(io.StringIO(pgn_text))
    if game is None:
        return []
    moves = [move.uci() for move in game.mainline_moves()]
    return moves


def remove_simple_variations(text: str) -> str:
    previous = None
    cleaned = text
    while previous != cleaned:
        previous = cleaned
        cleaned = VARIATION_RE.sub(' ', cleaned)
    return cleaned


def clean_san_token(token: str) -> Optional[str]:
    token = token.strip()
    token = re.sub(r'^\d+\.(?:\.\.)?', '', token)
    token = token.strip()
    if not token or token in RESULT_TOKENS or token == '...':
        return None
    token = re.sub(r'[!?]+$', '', token)
    token = token.replace('e.p.', '')
    return token or None


def pgn_text_to_uci_moves_fallback(pgn_text: str) -> List[str]:
    text = COMMENT_RE.sub(' ', str(pgn_text))
    text = remove_simple_variations(text)
    text = NAG_RE.sub(' ', text)
    text = text.replace('\n', ' ')

    board = chess.Board()
    moves = []
    for raw_token in text.split():
        token = clean_san_token(raw_token)
        if token is None:
            continue
        move = board.parse_san(token)
        moves.append(move.uci())
        board.push(move)
    return moves


def extract_uci_moves_from_row(row: pd.Series) -> List[str]:
    if isinstance(row.get('uci'), str) and row['uci'].strip():
        return row['uci'].split()

    if isinstance(row.get('pgn'), str) and row['pgn'].strip():
        parsed_moves = pgn_text_to_uci_moves_with_reader(row['pgn'])
        if parsed_moves:
            return parsed_moves
        return pgn_text_to_uci_moves_fallback(row['pgn'])

    return []


raw_df = load_source_dataframe(cfg)
display(raw_df.head())
print('Rows:', len(raw_df))

## Move Vocabulary and Board Encoding

The classifier predicts a UCI move class. Instead of building a vocabulary only from training data, the notebook builds a universal UCI-style move vocabulary over all from-square and to-square pairs plus promotion suffixes. This is intentionally broader than legal chess moves from a given position, but a legal-move mask is added to the logits so the model can only choose moves legal in the current board state.

Each board is encoded as two inputs:

- square_tokens: 64 piece IDs, one for each square in python-chess order from a1 to h8.
- global_features: side to move, castling rights, normalized halfmove clock, normalized fullmove number, and en passant file.

The fullmove feature is not used to decide which examples are included. The move-5 filter is applied earlier in the dataset builder. The feature merely tells the model where in the opening phase the board lies.

In [ ]:
PROMOTION_PIECES = ['q', 'r', 'b', 'n']


def build_universal_move_vocab() -> Tuple[List[str], Dict[str, int]]:
    move_set = set()
    for from_square in chess.SQUARES:
        from_name = chess.square_name(from_square)
        for to_square in chess.SQUARES:
            if from_square == to_square:
                continue
            to_name = chess.square_name(to_square)
            base = f'{from_name}{to_name}'
            move_set.add(base)
            if chess.square_rank(to_square) in (0, 7):
                for promo in PROMOTION_PIECES:
                    move_set.add(base + promo)
    vocab = sorted(move_set)
    move_to_idx = {move: idx for idx, move in enumerate(vocab)}
    return vocab, move_to_idx


MOVE_VOCAB, MOVE_TO_IDX = build_universal_move_vocab()
MOVE_VOCAB_SIZE = len(MOVE_VOCAB)
print('Universal move vocabulary size:', MOVE_VOCAB_SIZE)


def piece_to_id(piece: Optional[chess.Piece]) -> int:
    if piece is None:
        return 0
    offset = 0 if piece.color == chess.WHITE else 6
    return piece.piece_type + offset


GLOBAL_FEATURE_DIM = 15


def board_to_features(board: chess.Board) -> Tuple[List[int], List[float]]:
    square_tokens = [piece_to_id(board.piece_at(square)) for square in chess.SQUARES]
    ep_square = board.ep_square
    ep_file = chess.square_file(ep_square) if ep_square is not None else -1
    ep_one_hot = [1.0 if i == ep_file else 0.0 for i in range(8)]
    global_features = [
        float(board.turn == chess.WHITE),
        float(board.has_kingside_castling_rights(chess.WHITE)),
        float(board.has_queenside_castling_rights(chess.WHITE)),
        float(board.has_kingside_castling_rights(chess.BLACK)),
        float(board.has_queenside_castling_rights(chess.BLACK)),
        float(board.halfmove_clock) / 100.0,
        float(board.fullmove_number) / 100.0,
    ] + ep_one_hot
    return square_tokens, global_features


def legal_move_indices(board: chess.Board) -> List[int]:
    return [MOVE_TO_IDX[move.uci()] for move in board.legal_moves if move.uci() in MOVE_TO_IDX]

## Building Move-5 Examples and Tasks

An opening task is a group of examples that share an opening identifier. By default that identifier is the opening name, which creates fine-grained tasks such as particular Italian Game or Sicilian variations. Setting task_group_field to eco creates broader ECO-code tasks.

For every source line, the builder replays moves from the initial board. At each ply it first checks whether the target ply is at or after move 5. If it is, the current board is encoded and the current move becomes the label. Only after saving the example does the code push the move to advance the board. This order is what keeps positions and labels aligned.

Exact duplicate examples within a task are removed using the tuple of task_name, FEN, and target_uci. This reduces leakage from repeated opening prefixes appearing in multiple source lines. The support and query sets for an episode are then sampled as disjoint examples from the same held-out task.

In [ ]:
def safe_string(value, default: str) -> str:
    if value is None:
        return default
    try:
        if pd.isna(value):
            return default
    except TypeError:
        pass
    text = str(value).strip()
    return text if text else default


def normalize_row_id(row_id):
    try:
        return int(row_id)
    except Exception:
        return str(row_id)


def build_examples_from_dataframe(
    df: pd.DataFrame,
    group_field: str,
    max_plies: int,
    min_prediction_fullmove: int,
) -> List[Dict]:
    examples = []
    seen_keys = set()
    skip_counts = Counter()
    min_prediction_ply = first_ply_for_fullmove(min_prediction_fullmove)

    for row_id, row in tqdm(df.iterrows(), total=len(df), desc='Building move-5 examples'):
        try:
            uci_moves = extract_uci_moves_from_row(row)
        except Exception:
            skip_counts['parse_error'] += 1
            continue

        if len(uci_moves) <= min_prediction_ply:
            skip_counts['too_short_for_move_5'] += 1
            continue

        opening_name = safe_string(row.get('name', row.get('Opening')), 'Unknown Opening')
        eco = safe_string(row.get('eco', row.get('ECO')), 'UNK')
        task_name = safe_string(row.get(group_field), opening_name)
        source_row_id = normalize_row_id(row_id)

        board = chess.Board()
        line_limit = min(len(uci_moves), max_plies)
        for ply_idx, uci in enumerate(uci_moves[:line_limit]):
            try:
                move = chess.Move.from_uci(uci)
            except ValueError:
                skip_counts['invalid_uci_token'] += 1
                break

            if move not in board.legal_moves:
                skip_counts['illegal_move_sequence'] += 1
                break

            if ply_idx >= min_prediction_ply:
                square_tokens, global_features = board_to_features(board)
                legal_indices = legal_move_indices(board)
                target_index = MOVE_TO_IDX.get(uci)
                if target_index is None or target_index not in legal_indices:
                    skip_counts['target_not_in_legal_mask'] += 1
                else:
                    key = (task_name, board.fen(), uci)
                    if key in seen_keys:
                        skip_counts['duplicate_example'] += 1
                    else:
                        seen_keys.add(key)
                        examples.append(
                            {
                                'task_name': task_name,
                                'opening_name': opening_name,
                                'eco': eco,
                                'source_row_id': source_row_id,
                                'line_length_plies': len(uci_moves),
                                'ply_idx': ply_idx,
                                'ply_label': ply_label(ply_idx),
                                'target_fullmove_number': board.fullmove_number,
                                'side_to_move': 'white' if board.turn == chess.WHITE else 'black',
                                'fen': board.fen(),
                                'square_tokens': square_tokens,
                                'global_features': global_features,
                                'legal_indices': legal_indices,
                                'target_index': target_index,
                                'target_uci': uci,
                            }
                        )

            board.push(move)

    print('Built examples:', len(examples))
    print('Skipped rows/examples:', dict(skip_counts))
    return examples


def validate_example_alignment(examples: List[Dict], cfg: Config, max_checks: int = 1000) -> None:
    if not examples:
        raise ValueError('No examples were built.')

    rng = random.Random(SEED)
    sample_size = min(len(examples), max_checks)
    checked_examples = rng.sample(examples, sample_size)

    for ex in checked_examples:
        if ex['ply_idx'] < cfg.min_prediction_ply:
            raise AssertionError(f'Example before move 5 slipped through: {ex["ply_idx"]}')
        if ex['target_fullmove_number'] < cfg.min_prediction_fullmove:
            raise AssertionError(f'Fullmove filter failed: {ex["target_fullmove_number"]}')

        board = chess.Board(ex['fen'])
        target_move = chess.Move.from_uci(ex['target_uci'])
        if target_move not in board.legal_moves:
            raise AssertionError(f'Target move is illegal in stored FEN: {ex["target_uci"]} / {ex["fen"]}')
        if MOVE_TO_IDX[ex['target_uci']] != ex['target_index']:
            raise AssertionError('Stored target index does not match target UCI.')
        if ex['target_index'] not in ex['legal_indices']:
            raise AssertionError('Stored target index is missing from the legal mask.')
        if board.fullmove_number != ex['target_fullmove_number']:
            raise AssertionError('Stored fullmove number does not match FEN.')
        expected_side = 'white' if board.turn == chess.WHITE else 'black'
        if expected_side != ex['side_to_move']:
            raise AssertionError('Stored side_to_move does not match FEN.')

    print(f'Alignment checks passed for {sample_size} sampled examples.')


def group_examples_by_task(examples: List[Dict]) -> Dict[str, List[Dict]]:
    grouped = defaultdict(list)
    for example in examples:
        grouped[example['task_name']].append(example)
    return {task: sorted(task_examples, key=lambda ex: (str(ex['source_row_id']), ex['ply_idx'], ex['target_uci'])) for task, task_examples in grouped.items()}


def summarize_tasks(task_to_examples: Dict[str, List[Dict]]) -> pd.DataFrame:
    rows = []
    for task_name, task_examples in task_to_examples.items():
        rows.append(
            {
                'task_name': task_name,
                'examples': len(task_examples),
                'unique_targets': len({ex['target_uci'] for ex in task_examples}),
                'min_fullmove': min(ex['target_fullmove_number'] for ex in task_examples),
                'max_fullmove': max(ex['target_fullmove_number'] for ex in task_examples),
            }
        )
    return pd.DataFrame(rows).sort_values('examples', ascending=False).reset_index(drop=True)


def choose_holdout_task(task_to_examples: Dict[str, List[Dict]], requested_task: Optional[str]) -> str:
    if requested_task is not None:
        if requested_task not in task_to_examples:
            available = sorted(task_to_examples)[:10]
            raise ValueError(f'Requested holdout task not found. First tasks: {available}')
        return requested_task

    return max(task_to_examples, key=lambda name: len(task_to_examples[name]))


def split_train_validation_holdout_tasks(
    task_to_examples: Dict[str, List[Dict]],
    min_examples_per_task: int,
    requested_holdout_task: Optional[str],
    validation_task_count: int,
    seed: int = SEED,
) -> Tuple[Dict[str, List[Dict]], Dict[str, List[Dict]], str, List[Dict]]:
    filtered = {k: v for k, v in task_to_examples.items() if len(v) >= min_examples_per_task}
    if len(filtered) < 2:
        raise ValueError('Need at least two eligible tasks after filtering to evaluate a held-out opening.')

    holdout_task = choose_holdout_task(filtered, requested_holdout_task)
    remaining_tasks = [task for task in filtered if task != holdout_task]
    rng = random.Random(seed)
    rng.shuffle(remaining_tasks)

    actual_validation_count = min(validation_task_count, max(0, len(remaining_tasks) - 1))
    validation_names = set(remaining_tasks[:actual_validation_count])
    train_names = [task for task in remaining_tasks if task not in validation_names]

    if not train_names:
        raise ValueError('No training tasks remain after validation split. Reduce validation_task_count.')

    train_tasks = {task: filtered[task] for task in train_names}
    validation_tasks = {task: filtered[task] for task in validation_names}
    holdout_examples = filtered[holdout_task]
    return train_tasks, validation_tasks, holdout_task, holdout_examples


examples = build_examples_from_dataframe(
    raw_df,
    group_field=cfg.task_group_field,
    max_plies=cfg.max_plies,
    min_prediction_fullmove=cfg.min_prediction_fullmove,
)
validate_example_alignment(examples, cfg)

example_metadata_df = pd.DataFrame(
    [
        {
            'task_name': ex['task_name'],
            'opening_name': ex['opening_name'],
            'eco': ex['eco'],
            'ply_idx': ex['ply_idx'],
            'ply_label': ex['ply_label'],
            'target_fullmove_number': ex['target_fullmove_number'],
            'side_to_move': ex['side_to_move'],
            'target_uci': ex['target_uci'],
        }
        for ex in examples
    ]
)

task_to_examples = group_examples_by_task(examples)
task_summary_df = summarize_tasks(task_to_examples)
train_tasks, validation_tasks, holdout_task, holdout_examples = split_train_validation_holdout_tasks(
    task_to_examples,
    min_examples_per_task=cfg.min_examples_per_task,
    requested_holdout_task=cfg.holdout_task,
    validation_task_count=cfg.validation_task_count,
)

train_examples = [ex for task_examples in train_tasks.values() for ex in task_examples]
validation_examples = [ex for task_examples in validation_tasks.values() for ex in task_examples]

print('Total tasks:', len(task_to_examples))
print('Eligible train tasks:', len(train_tasks))
print('Validation tasks:', len(validation_tasks))
print('Train examples:', len(train_examples))
print('Validation examples:', len(validation_examples))
print('Held-out task:', holdout_task)
print('Held-out examples:', len(holdout_examples))
print('Minimum ply in all examples:', example_metadata_df['ply_idx'].min())
print('Minimum fullmove in all examples:', example_metadata_df['target_fullmove_number'].min())
display(task_summary_df.head(12))
display(example_metadata_df.head())

## Distribution Checks

Before training, inspect whether the generated dataset is balanced enough for the intended comparison. Opening data is naturally imbalanced: common opening families have many more lines than rare variations, and common moves such as castling or developing knights can dominate target labels.

The first plot below checks that no examples appear before move 5 and shows how many examples are available at each target fullmove number. The second plot shows the largest tasks by example count. A highly skewed task distribution can make both the baseline and MAML look better on frequent openings than on rare ones, so the final conclusions should be framed as prototype evidence rather than a definitive chess-opening benchmark.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

fullmove_counts = example_metadata_df['target_fullmove_number'].value_counts().sort_index()
fullmove_positions = np.arange(len(fullmove_counts))
axes[0].bar(fullmove_positions, fullmove_counts.values, color='#4C78A8')
axes[0].set_xticks(fullmove_positions)
axes[0].set_xticklabels(fullmove_counts.index.astype(str))
if cfg.min_prediction_fullmove in list(fullmove_counts.index):
    cutoff_idx = list(fullmove_counts.index).index(cfg.min_prediction_fullmove)
    axes[0].axvline(cutoff_idx - 0.5, color='#D62728', linestyle='--', linewidth=1.5, label='move-5 cutoff')
axes[0].set_title('Examples by target fullmove')
axes[0].set_xlabel('Target fullmove number')
axes[0].set_ylabel('Example count')
axes[0].legend()

top_tasks = task_summary_df.head(15).sort_values('examples')
axes[1].barh(top_tasks['task_name'], top_tasks['examples'], color='#F58518')
axes[1].set_title('Largest opening tasks after filtering')
axes[1].set_xlabel('Example count')
axes[1].set_ylabel('Task')
plt.show()

target_counts = example_metadata_df['target_uci'].value_counts().head(15).rename_axis('target_uci').reset_index(name='count')
display(target_counts)

## Model

Both methods use the same neural architecture so the comparison is about the training regime, not model capacity. The model is a compact Transformer encoder over the 64 board-square tokens. A learned CLS token receives the global board features, the Transformer produces a board representation, and a linear head scores every move in the universal move vocabulary.

A legal-move mask is added to the logits before the loss and accuracy are computed. Illegal moves receive a very negative logit, so top-1 accuracy means the model chose the exact labeled next move among legal moves from that board.

For fairness, the baseline model and MAML model are initialized from the same random state dictionary before their different training procedures begin.

In [ ]:
class FlatExampleDataset(Dataset):
    def __init__(self, examples: List[Dict]):
        self.examples = examples

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int) -> Dict:
        return self.examples[idx]


def collate_examples(examples: List[Dict], target_device: Optional[torch.device] = None) -> Dict[str, torch.Tensor]:
    square_tokens = torch.tensor([ex['square_tokens'] for ex in examples], dtype=torch.long)
    global_features = torch.tensor([ex['global_features'] for ex in examples], dtype=torch.float32)
    targets = torch.tensor([ex['target_index'] for ex in examples], dtype=torch.long)
    legal_mask = torch.full((len(examples), MOVE_VOCAB_SIZE), -1e9, dtype=torch.float32)
    for row_idx, ex in enumerate(examples):
        legal_mask[row_idx, ex['legal_indices']] = 0.0

    batch = {
        'square_tokens': square_tokens,
        'global_features': global_features,
        'targets': targets,
        'legal_mask': legal_mask,
    }
    if target_device is not None:
        batch = {k: v.to(target_device) for k, v in batch.items()}
    return batch


class ChessMoveTransformer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        global_feature_dim: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        ff_dim: int,
        dropout: float,
    ):
        super().__init__()
        self.piece_embed = nn.Embedding(13, d_model)
        self.square_embed = nn.Embedding(64, d_model)
        self.global_mlp = nn.Sequential(
            nn.Linear(global_feature_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, vocab_size),
        )

    def forward(
        self,
        square_tokens: torch.Tensor,
        global_features: torch.Tensor,
        legal_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        batch_size = square_tokens.size(0)
        square_positions = torch.arange(64, device=square_tokens.device).unsqueeze(0).expand(batch_size, -1)
        token_embeddings = self.piece_embed(square_tokens) + self.square_embed(square_positions)
        cls = self.cls_token.expand(batch_size, -1, -1) + self.global_mlp(global_features).unsqueeze(1)
        hidden = torch.cat([cls, token_embeddings], dim=1)
        encoded = self.encoder(hidden)
        logits = self.head(encoded[:, 0])
        if legal_mask is not None:
            logits = logits + legal_mask
        return logits


def build_model(cfg: Config) -> ChessMoveTransformer:
    return ChessMoveTransformer(
        vocab_size=MOVE_VOCAB_SIZE,
        global_feature_dim=GLOBAL_FEATURE_DIM,
        d_model=cfg.d_model,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ff_dim=cfg.ff_dim,
        dropout=cfg.dropout,
    )


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


initial_model = build_model(cfg).to(device)
initial_state = copy.deepcopy(initial_model.state_dict())

baseline_model = build_model(cfg).to(device)
baseline_model.load_state_dict(initial_state)

maml_model = build_model(cfg).to(device)
maml_model.load_state_dict(initial_state)

del initial_model, initial_state
print('Trainable parameters:', count_parameters(baseline_model))

sanity_examples = train_examples[: min(4, len(train_examples))]
sanity_batch = collate_examples(sanity_examples, target_device=device)
with torch.no_grad():
    sanity_logits = baseline_model(
        sanity_batch['square_tokens'],
        sanity_batch['global_features'],
        sanity_batch['legal_mask'],
    )
assert sanity_batch['square_tokens'].shape == (len(sanity_examples), 64)
assert sanity_batch['global_features'].shape[1] == GLOBAL_FEATURE_DIM
assert sanity_logits.shape == (len(sanity_examples), MOVE_VOCAB_SIZE)
print('Sanity batch shapes:', {
    'square_tokens': tuple(sanity_batch['square_tokens'].shape),
    'global_features': tuple(sanity_batch['global_features'].shape),
    'logits': tuple(sanity_logits.shape),
})

## Training and Evaluation Logic

The supervised baseline sees a flat dataset of positions from the training openings. During each epoch it shuffles examples, predicts the labeled next move, computes cross-entropy over legal moves, and updates model weights with AdamW. Its train loss and train accuracy are sample-weighted epoch averages. If validation tasks are available, the baseline is also evaluated on validation examples that were not used for training.

MAML is episodic. Each meta-step samples several opening tasks. For each task, it samples a support set and a query set. The inner loop adapts a temporary copy of the parameters using support loss. The outer update then uses query loss after adaptation to update the original initialization. This notebook uses first-order MAML, which ignores second-order derivatives for speed and stability.

The losses are not identical objectives. Baseline train loss is direct supervised loss before any few-shot adaptation. MAML meta loss is query loss after inner-loop adaptation. Their curves can both be useful, but they should not be interpreted as if they were the same measurement on the same x-axis.

In [ ]:
def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    predictions = logits.argmax(dim=-1)
    return (predictions == targets).float().mean()


def evaluate_model(model: nn.Module, examples: List[Dict], batch_size: int = 256) -> Dict[str, float]:
    if len(examples) == 0:
        return {'loss': math.nan, 'top1_accuracy': math.nan, 'n_examples': 0}

    loader = DataLoader(
        FlatExampleDataset(examples),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_examples,
    )
    model.eval()
    total_loss = 0.0
    total_correct = 0.0
    total_count = 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch['square_tokens'], batch['global_features'], batch['legal_mask'])
            loss = F.cross_entropy(logits, batch['targets'])
            batch_size_actual = batch['targets'].size(0)
            total_loss += loss.item() * batch_size_actual
            total_correct += (logits.argmax(dim=-1) == batch['targets']).sum().item()
            total_count += batch_size_actual
    return {
        'loss': total_loss / max(total_count, 1),
        'top1_accuracy': total_correct / max(total_count, 1),
        'n_examples': total_count,
    }


def train_supervised_baseline(
    model: nn.Module,
    train_examples: List[Dict],
    cfg: Config,
    val_examples: Optional[List[Dict]] = None,
) -> Dict[str, List[Dict[str, float]]]:
    loader = DataLoader(
        FlatExampleDataset(train_examples),
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_examples,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.baseline_lr, weight_decay=1e-4)
    epoch_history = []
    batch_history = []
    optimizer_step = 0
    num_batches = max(len(loader), 1)

    for epoch in range(1, cfg.baseline_epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0.0
        total_count = 0

        for batch_idx, batch in enumerate(tqdm(loader, desc=f'Baseline epoch {epoch}'), start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch['square_tokens'], batch['global_features'], batch['legal_mask'])
            loss = F.cross_entropy(logits, batch['targets'])
            loss.backward()
            if cfg.grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
            optimizer.step()

            optimizer_step += 1
            batch_size_actual = batch['targets'].size(0)
            batch_correct = (logits.argmax(dim=-1) == batch['targets']).sum().item()
            total_loss += loss.item() * batch_size_actual
            total_correct += batch_correct
            total_count += batch_size_actual

            should_log_batch = (
                optimizer_step == 1
                or optimizer_step % cfg.baseline_log_every_batches == 0
                or batch_idx == num_batches
            )
            if should_log_batch:
                batch_history.append(
                    {
                        'optimizer_step': optimizer_step,
                        'epoch': epoch,
                        'epoch_fraction': epoch - 1 + batch_idx / num_batches,
                        'train_loss': loss.item(),
                        'train_acc': batch_correct / max(batch_size_actual, 1),
                    }
                )

        epoch_stats = {
            'epoch': epoch,
            'optimizer_step': optimizer_step,
            'train_loss': total_loss / max(total_count, 1),
            'train_acc': total_correct / max(total_count, 1),
        }
        if val_examples:
            val_stats = evaluate_model(model, val_examples, batch_size=cfg.batch_size)
            epoch_stats.update(
                {
                    'val_loss': val_stats['loss'],
                    'val_acc': val_stats['top1_accuracy'],
                    'val_examples': val_stats['n_examples'],
                }
            )
        epoch_history.append(epoch_stats)
        print(epoch_stats)

    return {'epoch_history': epoch_history, 'batch_history': batch_history}


def sample_support_query(
    examples: List[Dict],
    support_size: int,
    query_size: int,
    rng: random.Random,
) -> Tuple[List[Dict], List[Dict]]:
    if len(examples) < support_size + query_size:
        raise ValueError('Task does not have enough examples for the requested support/query split.')
    chosen = rng.sample(examples, support_size + query_size)
    return chosen[:support_size], chosen[support_size:]


def inner_adapt(
    model: nn.Module,
    support_batch: Dict[str, torch.Tensor],
    inner_lr: float,
    inner_steps: int,
    first_order: bool = True,
) -> Dict[str, torch.Tensor]:
    adapted_params = {name: param for name, param in model.named_parameters()}
    for _ in range(inner_steps):
        support_logits = functional_call(
            model,
            adapted_params,
            (support_batch['square_tokens'], support_batch['global_features'], support_batch['legal_mask']),
        )
        support_loss = F.cross_entropy(support_logits, support_batch['targets'])
        grads = torch.autograd.grad(
            support_loss,
            tuple(adapted_params.values()),
            create_graph=not first_order,
        )
        adapted_params = {
            name: param - inner_lr * grad
            for (name, param), grad in zip(adapted_params.items(), grads)
        }
    return adapted_params


def meta_train_maml(model: nn.Module, train_tasks: Dict[str, List[Dict]], cfg: Config) -> List[Dict[str, float]]:
    eligible_tasks = [task for task, exs in train_tasks.items() if len(exs) >= cfg.support_size + cfg.query_size]
    if not eligible_tasks:
        raise ValueError('No tasks have enough examples for episodic training.')

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.outer_lr, weight_decay=1e-4)
    rng = random.Random(SEED)
    history = []

    for step in range(1, cfg.meta_steps + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        meta_loss = torch.zeros((), device=device)
        meta_acc_sum = 0.0

        for _ in range(cfg.meta_batch_size):
            task_name = rng.choice(eligible_tasks)
            support_examples, query_examples = sample_support_query(
                train_tasks[task_name],
                cfg.support_size,
                cfg.query_size,
                rng,
            )
            support_batch = collate_examples(support_examples, target_device=device)
            query_batch = collate_examples(query_examples, target_device=device)

            adapted_params = inner_adapt(
                model,
                support_batch,
                inner_lr=cfg.inner_lr,
                inner_steps=cfg.inner_steps,
                first_order=True,
            )
            query_logits = functional_call(
                model,
                adapted_params,
                (query_batch['square_tokens'], query_batch['global_features'], query_batch['legal_mask']),
            )
            query_loss = F.cross_entropy(query_logits, query_batch['targets'])
            meta_loss = meta_loss + query_loss / cfg.meta_batch_size
            meta_acc_sum += accuracy_from_logits(query_logits, query_batch['targets']).item()

        meta_loss.backward()
        if cfg.grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        optimizer.step()

        stats = {
            'step': step,
            'episodes_seen': step * cfg.meta_batch_size,
            'query_examples_seen': step * cfg.meta_batch_size * cfg.query_size,
            'meta_loss': float(meta_loss.item()),
            'meta_acc': float(meta_acc_sum / cfg.meta_batch_size),
        }
        history.append(stats)
        if step == 1 or step % cfg.maml_log_every_steps == 0:
            print(stats)

    return history

In [ ]:
def adapt_standard_model(model: nn.Module, support_examples: List[Dict], cfg: Config) -> nn.Module:
    adapted_model = copy.deepcopy(model).to(device)
    optimizer = torch.optim.AdamW(adapted_model.parameters(), lr=cfg.adaptation_lr, weight_decay=1e-4)
    support_batch = collate_examples(support_examples, target_device=device)
    adapted_model.train()
    for _ in range(cfg.adaptation_steps):
        optimizer.zero_grad(set_to_none=True)
        logits = adapted_model(
            support_batch['square_tokens'],
            support_batch['global_features'],
            support_batch['legal_mask'],
        )
        loss = F.cross_entropy(logits, support_batch['targets'])
        loss.backward()
        if cfg.grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(adapted_model.parameters(), cfg.grad_clip_norm)
        optimizer.step()
    return adapted_model


def evaluate_few_shot_baseline(
    model: nn.Module,
    support_examples: List[Dict],
    query_examples: List[Dict],
    cfg: Config,
) -> Dict[str, Dict[str, float]]:
    before = evaluate_model(model, query_examples, batch_size=cfg.batch_size)
    adapted_model = adapt_standard_model(model, support_examples, cfg)
    after = evaluate_model(adapted_model, query_examples, batch_size=cfg.batch_size)
    return {'Before adaptation': before, 'After adaptation': after}


def evaluate_few_shot_maml(
    model: nn.Module,
    support_examples: List[Dict],
    query_examples: List[Dict],
    cfg: Config,
) -> Dict[str, Dict[str, float]]:
    support_batch = collate_examples(support_examples, target_device=device)
    query_batch = collate_examples(query_examples, target_device=device)

    model.eval()
    with torch.no_grad():
        pre_logits = model(query_batch['square_tokens'], query_batch['global_features'], query_batch['legal_mask'])
        before = {
            'loss': F.cross_entropy(pre_logits, query_batch['targets']).item(),
            'top1_accuracy': accuracy_from_logits(pre_logits, query_batch['targets']).item(),
            'n_examples': len(query_examples),
        }

    adapted_params = inner_adapt(
        model,
        support_batch,
        inner_lr=cfg.inner_lr,
        inner_steps=cfg.inner_steps,
        first_order=True,
    )

    with torch.no_grad():
        post_logits = functional_call(
            model,
            adapted_params,
            (query_batch['square_tokens'], query_batch['global_features'], query_batch['legal_mask']),
        )
        after = {
            'loss': F.cross_entropy(post_logits, query_batch['targets']).item(),
            'top1_accuracy': accuracy_from_logits(post_logits, query_batch['targets']).item(),
            'n_examples': len(query_examples),
        }

    return {'Before adaptation': before, 'After adaptation': after}


def build_holdout_episode(
    holdout_examples: List[Dict],
    support_size: int,
    query_size: int,
    seed: int = SEED,
) -> Tuple[List[Dict], List[Dict]]:
    rng = random.Random(seed)
    return sample_support_query(holdout_examples, support_size, query_size, rng)


def evaluate_many_holdout_episodes(
    baseline_model: nn.Module,
    maml_model: nn.Module,
    holdout_examples: List[Dict],
    cfg: Config,
) -> pd.DataFrame:
    rng = random.Random(SEED + 1000)
    rows = []
    for episode_idx in tqdm(range(cfg.num_eval_episodes), desc='Held-out few-shot episodes'):
        episode_seed = rng.randint(0, 10**9)
        support_examples, query_examples = build_holdout_episode(
            holdout_examples,
            support_size=cfg.support_size,
            query_size=cfg.query_size,
            seed=episode_seed,
        )
        episode_payload = {
            'Baseline': evaluate_few_shot_baseline(baseline_model, support_examples, query_examples, cfg),
            'MAML': evaluate_few_shot_maml(maml_model, support_examples, query_examples, cfg),
        }
        for model_name, model_results in episode_payload.items():
            for stage, metrics in model_results.items():
                rows.append(
                    {
                        'episode': episode_idx,
                        'episode_seed': episode_seed,
                        'model': model_name,
                        'stage': stage,
                        'top1_accuracy': metrics['top1_accuracy'],
                        'cross_entropy_loss': metrics['loss'],
                        'query_examples': metrics['n_examples'],
                    }
                )
    return pd.DataFrame(rows)


def summarize_episode_results(episode_results_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model_name, stage), group in episode_results_df.groupby(['model', 'stage'], sort=False):
        row = {
            'model': model_name,
            'stage': stage,
            'n_episodes': len(group),
        }
        for metric in ['top1_accuracy', 'cross_entropy_loss']:
            values = group[metric].astype(float)
            mean = values.mean()
            std = values.std(ddof=1) if len(values) > 1 else 0.0
            sem = std / math.sqrt(len(values)) if len(values) > 1 else 0.0
            row[f'{metric}_mean'] = mean
            row[f'{metric}_std'] = std
            row[f'{metric}_sem'] = sem
            row[f'{metric}_ci95'] = 1.96 * sem
        rows.append(row)
    return pd.DataFrame(rows)

## Train the Supervised Baseline

One baseline epoch means one pass over the flat training-example dataset. The x-axis for baseline epoch plots is therefore an epoch count, not a meta-step count. The loss is the mean supervised cross-entropy for predicting the line's next move from the current board. The accuracy is top-1 exact-match accuracy after the legal-move mask is applied.

The validation columns, when present, evaluate on validation openings that were not used for baseline training. This is a sanity check for generalization, not the final held-out few-shot result.

In [ ]:
baseline_train = train_supervised_baseline(
    baseline_model,
    train_examples,
    cfg,
    val_examples=validation_examples,
)
baseline_history = baseline_train['epoch_history']
baseline_batch_history = baseline_train['batch_history']
baseline_history_df = pd.DataFrame(baseline_history)
baseline_batch_history_df = pd.DataFrame(baseline_batch_history)
display(baseline_history_df)

## Train the MAML Meta-Learner

One MAML meta-step is one outer update. Inside that meta-step, the model samples meta_batch_size opening tasks. For each sampled task it adapts on support_size examples and computes the meta-objective on query_size different examples from the same task.

The MAML meta loss plotted later is the average query cross-entropy after adaptation. The MAML meta accuracy is the average query top-1 accuracy after adaptation. These are logged every meta-step in memory and printed every maml_log_every_steps steps.

In [ ]:
maml_history = meta_train_maml(maml_model, train_tasks, cfg)
maml_history_df = pd.DataFrame(maml_history)
display(maml_history_df.tail())

## Held-Out Few-Shot Evaluation

The held-out opening task is excluded from both supervised baseline training and MAML meta-training. Each evaluation episode samples support examples and query examples from that held-out opening. The support set is used only for adaptation. The query set is used only for evaluation.

The notebook evaluates multiple held-out episodes rather than a single support/query split. A single four-example query set can swing by 25 percentage points per correct prediction, so reporting a mean and confidence interval across episodes is much more informative than one bar from one random episode.

In [ ]:
support_examples, query_examples = build_holdout_episode(
    holdout_examples,
    support_size=cfg.support_size,
    query_size=cfg.query_size,
)

print('Held-out task:', holdout_task)
print('Support example count:', len(support_examples))
print('Query example count:', len(query_examples))
print('Support ply labels:', [ex['ply_label'] for ex in support_examples])
print('Query ply labels:', [ex['ply_label'] for ex in query_examples])
print('Minimum held-out support/query fullmove:', min(ex['target_fullmove_number'] for ex in support_examples + query_examples))

episode_results_df = evaluate_many_holdout_episodes(baseline_model, maml_model, holdout_examples, cfg)
results_summary_df = summarize_episode_results(episode_results_df)
results_df = results_summary_df.copy()

display(episode_results_df.head())
display(results_summary_df)

## Graphs and Metric Interpretation

The training-curve figure uses separate x-axes because the baseline and MAML histories are logged on different clocks.

- Baseline panels use epoch on the x-axis. One epoch is one pass over the supervised training examples.
- MAML panels use outer update, also called meta-step, on the x-axis. One meta-step contains many sampled support/query episodes.
- Loss y-axes are cross-entropy, but the objectives differ: baseline loss is direct supervised training loss, while MAML loss is query loss after inner-loop adaptation.
- Accuracy y-axes are top-1 exact-match accuracy after legal-move masking.

The normalized-progress figure is only a visual aid. Its x-axis is the fraction of the configured training budget completed. It helps compare overall convergence shapes without pretending that one baseline epoch equals one MAML meta-step.

The held-out results figure reports means across evaluation episodes with 95 percent confidence intervals. Individual episode points are overlaid to show how noisy few-shot evaluation can be with small query sets.

In [ ]:
MODEL_ORDER = ['Baseline', 'MAML']
STAGE_ORDER = ['Before adaptation', 'After adaptation']
COLORS = {'Baseline': '#4C78A8', 'MAML': '#F58518'}


def smooth_series(values, window: int = 10) -> pd.Series:
    return pd.Series(values).rolling(window=window, min_periods=1).mean()


fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)

axes[0, 0].plot(baseline_history_df['epoch'], baseline_history_df['train_loss'], marker='o', label='Train')
if 'val_loss' in baseline_history_df and baseline_history_df['val_loss'].notna().any():
    axes[0, 0].plot(baseline_history_df['epoch'], baseline_history_df['val_loss'], marker='s', label='Validation')
axes[0, 0].set_title('Supervised baseline loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Cross-entropy loss')
axes[0, 0].legend()

axes[1, 0].plot(baseline_history_df['epoch'], baseline_history_df['train_acc'], marker='o', label='Train')
if 'val_acc' in baseline_history_df and baseline_history_df['val_acc'].notna().any():
    axes[1, 0].plot(baseline_history_df['epoch'], baseline_history_df['val_acc'], marker='s', label='Validation')
axes[1, 0].set_title('Supervised baseline accuracy')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Top-1 accuracy')
axes[1, 0].set_ylim(0, 1)
axes[1, 0].legend()

maml_loss_smooth = smooth_series(maml_history_df['meta_loss'], window=10)
maml_acc_smooth = smooth_series(maml_history_df['meta_acc'], window=10)

axes[0, 1].plot(maml_history_df['step'], maml_history_df['meta_loss'], color=COLORS['MAML'], alpha=0.25, label='Per-step')
axes[0, 1].plot(maml_history_df['step'], maml_loss_smooth, color=COLORS['MAML'], label='10-step moving average')
axes[0, 1].set_title('MAML meta loss')
axes[0, 1].set_xlabel('Outer update / meta-step')
axes[0, 1].set_ylabel('Query cross-entropy after adaptation')
axes[0, 1].legend()

axes[1, 1].plot(maml_history_df['step'], maml_history_df['meta_acc'], color=COLORS['MAML'], alpha=0.25, label='Per-step')
axes[1, 1].plot(maml_history_df['step'], maml_acc_smooth, color=COLORS['MAML'], label='10-step moving average')
axes[1, 1].set_title('MAML meta accuracy')
axes[1, 1].set_xlabel('Outer update / meta-step')
axes[1, 1].set_ylabel('Query top-1 accuracy after adaptation')
axes[1, 1].set_ylim(0, 1)
axes[1, 1].legend()

plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

baseline_progress = baseline_history_df['epoch'] / cfg.baseline_epochs
maml_progress = maml_history_df['step'] / cfg.meta_steps

axes[0].plot(baseline_progress, baseline_history_df['train_loss'], marker='o', color=COLORS['Baseline'], label='Baseline supervised loss')
axes[0].plot(maml_progress, maml_loss_smooth, color=COLORS['MAML'], label='MAML adapted query loss')
axes[0].set_title('Loss by normalized training progress')
axes[0].set_xlabel('Fraction of configured training completed')
axes[0].set_ylabel('Cross-entropy loss, different objectives')
axes[0].legend()

axes[1].plot(baseline_progress, baseline_history_df['train_acc'], marker='o', color=COLORS['Baseline'], label='Baseline supervised accuracy')
axes[1].plot(maml_progress, maml_acc_smooth, color=COLORS['MAML'], label='MAML adapted query accuracy')
axes[1].set_title('Accuracy by normalized training progress')
axes[1].set_xlabel('Fraction of configured training completed')
axes[1].set_ylabel('Top-1 accuracy')
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.show()

In [ ]:
def plot_heldout_metric(summary_df: pd.DataFrame, episode_df: pd.DataFrame, metric: str, ylabel: str, ax) -> None:
    x = np.arange(len(STAGE_ORDER))
    width = 0.36
    rng = np.random.default_rng(SEED)

    for model_idx, model_name in enumerate(MODEL_ORDER):
        offset = (model_idx - 0.5) * width
        means = []
        errors = []
        for stage in STAGE_ORDER:
            row = summary_df[(summary_df['model'] == model_name) & (summary_df['stage'] == stage)]
            means.append(float(row[f'{metric}_mean'].iloc[0]))
            errors.append(float(row[f'{metric}_ci95'].iloc[0]))
        ax.bar(
            x + offset,
            means,
            width=width,
            yerr=errors,
            capsize=4,
            label=model_name,
            color=COLORS[model_name],
            alpha=0.85,
            edgecolor='black',
            linewidth=0.6,
        )

        for stage_idx, stage in enumerate(STAGE_ORDER):
            values = episode_df[(episode_df['model'] == model_name) & (episode_df['stage'] == stage)][metric].astype(float).values
            jitter = rng.normal(loc=0.0, scale=0.018, size=len(values))
            ax.scatter(
                np.full(len(values), x[stage_idx] + offset) + jitter,
                values,
                s=16,
                color='black',
                alpha=0.35,
                linewidth=0,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(STAGE_ORDER)
    ax.set_ylabel(ylabel)
    ax.legend()


fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
plot_heldout_metric(results_summary_df, episode_results_df, 'top1_accuracy', 'Top-1 accuracy', axes[0])
axes[0].set_title('Held-out opening accuracy')
axes[0].set_ylim(0, 1)

plot_heldout_metric(results_summary_df, episode_results_df, 'cross_entropy_loss', 'Cross-entropy loss', axes[1])
axes[1].set_title('Held-out opening loss')
axes[1].set_ylim(bottom=0)
plt.show()

improvement_rows = []
for (episode, model_name), group in episode_results_df.groupby(['episode', 'model']):
    before = group[group['stage'] == 'Before adaptation']['top1_accuracy'].iloc[0]
    after = group[group['stage'] == 'After adaptation']['top1_accuracy'].iloc[0]
    improvement_rows.append({'episode': episode, 'model': model_name, 'adaptation_gain': after - before})
adaptation_gain_df = pd.DataFrame(improvement_rows)

gain_summary = adaptation_gain_df.groupby('model')['adaptation_gain'].agg(['mean', 'std', 'count']).reset_index()
gain_summary['sem'] = gain_summary['std'] / np.sqrt(gain_summary['count'])
gain_summary['ci95'] = 1.96 * gain_summary['sem']
display(gain_summary)

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
for idx, model_name in enumerate(MODEL_ORDER):
    values = adaptation_gain_df[adaptation_gain_df['model'] == model_name]['adaptation_gain'].values
    error = 1.96 * values.std(ddof=1) / math.sqrt(len(values)) if len(values) > 1 else 0.0
    ax.bar(idx, values.mean(), yerr=error, capsize=4, color=COLORS[model_name], edgecolor='black', linewidth=0.6)
    jitter = np.random.default_rng(SEED + idx).normal(0.0, 0.035, size=len(values))
    ax.scatter(np.full(len(values), idx) + jitter, values, color='black', alpha=0.4, s=18)
ax.axhline(0.0, color='black', linewidth=1)
ax.set_xticks(range(len(MODEL_ORDER)))
ax.set_xticklabels(MODEL_ORDER)
ax.set_ylabel('After accuracy minus before accuracy')
ax.set_title('Few-shot adaptation gain on held-out opening')
plt.show()

## Automatic Result Interpretation

The next cell prints a short interpretation from the measured held-out episode table. A credible claim that MAML outperforms the supervised baseline should be based on post-adaptation held-out performance, not on training loss. The claim is stronger when the MAML mean is higher, the confidence intervals are narrow, and adaptation gains are consistently positive across episodes.

Because this notebook uses small support and query sets by default, the confidence intervals are part of the result. If they overlap heavily, the right conclusion is not that the methods are tied forever, but that this run does not provide strong enough evidence to separate them.

In [ ]:
def get_summary_value(summary_df: pd.DataFrame, model_name: str, stage: str, column: str) -> float:
    row = summary_df[(summary_df['model'] == model_name) & (summary_df['stage'] == stage)]
    return float(row[column].iloc[0])


def print_result_interpretation(summary_df: pd.DataFrame, gain_df: pd.DataFrame) -> None:
    base_post = get_summary_value(summary_df, 'Baseline', 'After adaptation', 'top1_accuracy_mean')
    maml_post = get_summary_value(summary_df, 'MAML', 'After adaptation', 'top1_accuracy_mean')
    base_ci = get_summary_value(summary_df, 'Baseline', 'After adaptation', 'top1_accuracy_ci95')
    maml_ci = get_summary_value(summary_df, 'MAML', 'After adaptation', 'top1_accuracy_ci95')
    delta = maml_post - base_post

    print(f'Post-adaptation top-1 accuracy: Baseline = {base_post:.3f} +/- {base_ci:.3f}, MAML = {maml_post:.3f} +/- {maml_ci:.3f}.')
    print(f'MAML minus baseline after adaptation: {delta:+.3f}.')

    for model_name in MODEL_ORDER:
        values = gain_df[gain_df['model'] == model_name]['adaptation_gain']
        print(f'{model_name} mean adaptation gain: {values.mean():+.3f} across {len(values)} episodes.')

    if delta > 0:
        print('In this run, MAML has the higher mean held-out post-adaptation accuracy.')
    elif delta < 0:
        print('In this run, the supervised baseline has the higher mean held-out post-adaptation accuracy.')
    else:
        print('In this run, the two methods have identical mean held-out post-adaptation accuracy.')

    print('Interpret this together with the confidence intervals, the episode scatter, and the class/task imbalance checks above.')


print_result_interpretation(results_summary_df, adaptation_gain_df)

## Scientific Limitations and Conclusions

This notebook is now set up as a careful prototype, but not as a final benchmark paper. The most important limitations are:

- Opening labels are derived from curated lines, not from independent human games. Many lines share prefixes, so deduplication is necessary but does not remove all correlation.
- A task is an opening name or ECO code, depending on configuration. Fine-grained names produce more specific tasks, while ECO codes produce broader and easier tasks. Results should state which grouping was used.
- The automatic holdout task is chosen because it has many examples. This improves evaluation stability but can bias the experiment toward common openings. For a stronger study, repeat the experiment over several held-out openings.
- Top-1 accuracy is harsh. In chess openings, multiple legal moves can be theoretically sound even if only one move appears in the source line. Cross-entropy and top-k accuracy can be added for a more nuanced analysis.
- Few-shot evaluation with support_size = 4 and query_size = 4 is noisy. The repeated-episode confidence intervals help, but larger query sets or more held-out tasks would be scientifically stronger.
- MAML and the baseline optimize different objectives. Training curves diagnose optimization, while held-out post-adaptation performance is the actual comparison.

The core conclusion should be phrased conditionally: MAML outperforms the supervised baseline only if its held-out post-adaptation accuracy is meaningfully higher across repeated episodes and not just higher on one random support/query split. If the confidence intervals overlap or adaptation gains are inconsistent, the honest conclusion is that this experiment does not yet show a reliable MAML advantage.

## How to Adapt This Notebook

- To reserve a specific opening for zero-shot testing, set cfg.holdout_task to the exact opening name or ECO code.
- To group tasks more coarsely, switch cfg.task_group_field from 'name' to 'eco'.
- To use raw Lichess games instead of curated opening lines, upload a PGN file and set cfg.data_source = 'pgn_file'.
- To train longer, increase cfg.baseline_epochs and cfg.meta_steps.
- To make evaluation more stable, increase cfg.num_eval_episodes or cfg.query_size.
- To study the move cutoff, change cfg.min_prediction_fullmove, but keep the validation cell enabled so ply and fullmove alignment errors are caught immediately.